<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [19]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests

# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd

# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices.
import numpy as np

# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all columns of a dataframe
pd.set_option('display.max_columns', None)

# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

# Inizializzazione delle liste globali - VERIFICA CHE TUTTE LE PARENTESI SIANO CHIUSE
BoosterVersion = []
Longitude = []
Latitude = []
LaunchSite = []
PayloadMass = []
Orbit = []
Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [20]:
# Takes the dataset and uses the rocket column to call the API and append the data_to_the_list
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
            BoosterVersion.append(response['name'])

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [21]:
# Takes the dataset and uses the launchpad column to call the API and append the data_to_the_list
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [22]:
# Takes the dataset and uses the payloads column to call the API and append the data_to_the_lists
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [23]:
# Takes the dataset and uses the cores column to call the API and append the data_to_the_lists
def getCoreData(data):
    for core in data['cores']:
        if core['core']:  # Verifica se core['core'] esiste e non è None/empty
            response = requests.get("https://api.spacexdata.com/v4/cores/" + core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [24]:
# Now let's start requesting rocket launch data from SpaceX API
spacex_url = "https://api.spacexdata.com/v4/launches/past"

In [25]:
response = requests.get(spacex_url)

Check the content of the response


In [29]:
# Le funzioni devono iterare su ogni elemento della lista
def getBoosterVersion(data):
    for launch in data:  # Itera su ogni lancio nella lista
        rocket_id = launch['rocket']
        if rocket_id:
            response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(rocket_id)).json()
            BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for launch in data:  # Itera su ogni lancio
        launchpad_id = launch['launchpad']
        if launchpad_id:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(launchpad_id)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

def getPayloadData(data):
    for launch in data:  # Itera su ogni lancio
        for payload_id in launch['payloads']:  # Itera su ogni payload del lancio
            if payload_id:
                response = requests.get("https://api.spacexdata.com/v4/payloads/"+payload_id).json()
                PayloadMass.append(response['mass_kg'])
                Orbit.append(response['orbit'])

def getCoreData(data):
    for launch in data:  # Itera su ogni lancio
        for core in launch['cores']:  # Itera su ogni core del lancio
            if core['core']:
                response = requests.get("https://api.spacexdata.com/v4/cores/" + core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

# Ora usa le funzioni
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()  # Questa è una LISTA di lanci

# Inizializza le liste
BoosterVersion = []
Longitude = []
Latitude = []
LaunchSite = []
PayloadMass = []
Orbit = []
Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []

# Ora chiama le funzioni
getBoosterVersion(launch_data)
getLaunchSite(launch_data)
getPayloadData(launch_data)
getCoreData(launch_data)

# Verifica i risultati
print(f"Numero di booster: {len(BoosterVersion)}")
print(f"Numero di siti di lancio: {len(LaunchSite)}")
print(f"Numero di payload: {len(PayloadMass)}")
print(f"Numero di core: {len(Block)}")

Numero di booster: 187
Numero di siti di lancio: 187
Numero di payload: 216
Numero di core: 193


You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [30]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [31]:
response=requests.get(static_json_url)

In [32]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [36]:
# Use json_normalize meethod to convert the json result into a dataframe
# Usa json_normalize per convertire il risultato JSON in un dataframe (metodo corretto)
import pandas as pd
import json

# Assumendo che launch_data sia già stato ottenuto con:
# response = requests.get("https://api.spacexdata.com/v4/launches/past")
# launch_data = response.json()

# Usa json_normalize per appiattire la struttura JSON
df = pd.json_normalize(launch_data)

# Ora visualizza il dataframe correttamente
print("Dataframe con json_normalize:")
print(df.head())

# Per vedere meglio alcune colonne specifiche
print("\nColonne disponibili:")
print(df.columns.tolist())

# Visualizza solo alcune colonne rilevanti
print("\nPrime 5 righe (solo colonne principali):")
cols_to_show = ['flight_number', 'name', 'date_utc', 'rocket', 'success', 'details']
available_cols = [col for col in cols_to_show if col in df.columns]
print(df[available_cols].head())

Dataframe con json_normalize:
       static_fire_date_utc  static_fire_date_unix    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False     0.0   
1                      None                    NaN  False     0.0   
2                      None                    NaN  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False     0.0   
4                      None                    NaN  False     0.0   

                     rocket success  \
0  5e9d0d95eda69955f709d1eb   False   
1  5e9d0d95eda69955f709d1eb   False   
2  5e9d0d95eda69955f709d1eb   False   
3  5e9d0d95eda69955f709d1eb    True   
4  5e9d0d95eda69955f709d1eb    True   

                                                                                                            failures  \
0                                                [{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]   
1            [{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation lea

Using the dataframe <code>data</code> print the first 5 rows


In [37]:
# Get the head of the dataframe
import pandas as pd
import requests

# Ottieni i dati dall'API SpaceX
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# Converti in dataframe usando json_normalize
df = pd.json_normalize(launch_data)

# Stampa le prime 5 righe
print("PRIME 5 RIGHE DEL DATAFRAME:")
print("=" * 80)
print(df.head())

# Per una visualizzazione più chiara, puoi anche selezionare colonne specifiche
print("\n\nPRIME 5 RIGHE (colonne principali):")
print("=" * 80)
cols_to_show = ['flight_number', 'name', 'date_utc', 'rocket', 'success']
print(df[cols_to_show].head())

# Se vuoi vedere anche i dettagli dei core
print("\n\nPRIME 5 RIGHE CON DETTAGLI CORES:")
print("=" * 80)
# Estrai il primo core da ogni lancio per semplicità
df['first_core_id'] = df['cores'].apply(lambda x: x[0]['core'] if x and len(x) > 0 else None)
df['first_core_flight'] = df['cores'].apply(lambda x: x[0]['flight'] if x and len(x) > 0 else None)

detailed_cols = ['flight_number', 'name', 'date_utc', 'success', 'first_core_id', 'first_core_flight']
print(df[detailed_cols].head())

PRIME 5 RIGHE DEL DATAFRAME:
       static_fire_date_utc  static_fire_date_unix    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False     0.0   
1                      None                    NaN  False     0.0   
2                      None                    NaN  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False     0.0   
4                      None                    NaN  False     0.0   

                     rocket success  \
0  5e9d0d95eda69955f709d1eb   False   
1  5e9d0d95eda69955f709d1eb   False   
2  5e9d0d95eda69955f709d1eb   False   
3  5e9d0d95eda69955f709d1eb    True   
4  5e9d0d95eda69955f709d1eb    True   

                                                                                                            failures  \
0                                                [{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]   
1            [{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation lead

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [39]:
import requests
import pandas as pd
import datetime

# 1. Ottieni i dati dall'API
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# 2. Converti in dataframe con json_normalize
data = pd.json_normalize(launch_data)

# 3. Seleziona le colonne che ci interessano
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# 4. Filtra per mantenere solo lanci con un core e un payload
print(f"Righe prima del filtro: {len(data)}")
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
print(f"Righe dopo filtro core/payload singoli: {len(data)}")

# 5. Estrai i valori dalle liste
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# 6. Converti date_utc in datetime e crea colonna date
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# 7. Filtra per data
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(f"Righe dopo filtro data: {len(data)}")

# 8. Visualizza risultato
print("\nPrime 5 righe del dataframe filtrato:")
print(data.head())

# 9. Verifica i tipi di dato
print("\nTipi di dato:")
print(data.dtypes)

Righe prima del filtro: 187
Righe dopo filtro core/payload singoli: 172
Righe dopo filtro data: 94

Prime 5 righe del dataframe filtrato:
                     rocket                  payloads  \
0  5e9d0d95eda69955f709d1eb  5eb0e4b5b6c3bb0006eeb1e1   
1  5e9d0d95eda69955f709d1eb  5eb0e4b6b6c3bb0006eeb1e2   
3  5e9d0d95eda69955f709d1eb  5eb0e4b7b6c3bb0006eeb1e5   
4  5e9d0d95eda69955f709d1eb  5eb0e4b7b6c3bb0006eeb1e6   
5  5e9d0d95eda69973a809d1ec  5eb0e4b7b6c3bb0006eeb1e7   

                  launchpad  \
0  5e9e4502f5090995de566f86   
1  5e9e4502f5090995de566f86   
3  5e9e4502f5090995de566f86   
4  5e9e4502f5090995de566f86   
5  5e9e4501f509094ba4566f84   

                                                                                                                                                                                            cores  \
0  {'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'land

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [40]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [41]:
BoosterVersion

[]

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [46]:
def getBoosterVersion(data):
    print("\n--- getBoosterVersion ---")
    count = 0
    for launch in data:
        rocket_id = launch['rocket']
        if rocket_id:
            try:
                url = f"https://api.spacexdata.com/v4/rockets/{rocket_id}"
                response = requests.get(url)
                if response.status_code == 200:
                    BoosterVersion.append(response.json().get('name'))
                    count += 1
                else:
                    BoosterVersion.append(None)
            except:
                BoosterVersion.append(None)
        else:
            BoosterVersion.append(None)
    print(f"Elaborati {count} booster")

def getLaunchSite(data):
    print("\n--- getLaunchSite ---")
    count = 0
    for launch in data:
        launchpad_id = launch['launchpad']
        if launchpad_id:
            try:
                url = f"https://api.spacexdata.com/v4/launchpads/{launchpad_id}"
                response = requests.get(url)
                if response.status_code == 200:
                    response_json = response.json()
                    Longitude.append(response_json.get('longitude'))
                    Latitude.append(response_json.get('latitude'))
                    LaunchSite.append(response_json.get('name'))
                    count += 1
                else:
                    Longitude.append(None)
                    Latitude.append(None)
                    LaunchSite.append(None)
            except:
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append(None)
        else:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)
    print(f"Elaborati {count} launchpad")

def getPayloadData(data):
    print("\n--- getPayloadData ---")
    count = 0
    for launch in data:
        payload_id = launch['payloads']  # Ora è una stringa singola
        if payload_id:
            try:
                url = f"https://api.spacexdata.com/v4/payloads/{payload_id}"
                response = requests.get(url)
                if response.status_code == 200:
                    response_json = response.json()
                    PayloadMass.append(response_json.get('mass_kg'))
                    Orbit.append(response_json.get('orbit'))
                    count += 1
                else:
                    PayloadMass.append(None)
                    Orbit.append(None)
            except:
                PayloadMass.append(None)
                Orbit.append(None)
        else:
            PayloadMass.append(None)
            Orbit.append(None)
    print(f"Elaborati {count} payload")

def getCoreData(data):
    print("\n--- getCoreData ---")
    count = 0
    for launch in data:
        core = launch['cores']  # Ora è un dizionario
        if core and core.get('core'):
            try:
                url = f"https://api.spacexdata.com/v4/cores/{core['core']}"
                response = requests.get(url)
                if response.status_code == 200:
                    response_json = response.json()
                    Block.append(response_json.get('block'))
                    ReusedCount.append(response_json.get('reuse_count'))
                    Serial.append(response_json.get('serial'))
                    count += 1
                else:
                    Block.append(None)
                    ReusedCount.append(None)
                    Serial.append(None)
            except:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        
        # Questi dati vengono sempre dal core locale
        Outcome.append(str(core.get('landing_success')) + ' ' + str(core.get('landing_type')))
        Flights.append(core.get('flight'))
        GridFins.append(core.get('gridfins'))
        Reused.append(core.get('reused'))
        Legs.append(core.get('legs'))
        LandingPad.append(core.get('landpad'))
    
    print(f"Elaborati {count} core con dettagli API")
    print(f"Totale core processati: {len(Block)}")

the list has now been update 


In [47]:
BoosterVersion[0:5]

['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']

we can apply the rest of the  functions here:


In [49]:
import pandas as pd
import requests
import datetime

# ... (il tuo codice precedente per ottenere e filtrare i dati) ...

# Dopo aver filtrato il dataframe
print("DataFrame shape:", data.shape)
print("Tipo di data:", type(data))

# Converti il DataFrame in lista di dizionari
data_list = data.to_dict('records')
print("Tipo di data_list:", type(data_list))
print("Numero elementi in data_list:", len(data_list))
print("Primo elemento di data_list (esempio):")
print(data_list[0] if data_list else "Lista vuota")

# Inizializza le liste
BoosterVersion = []
Longitude = []
Latitude = []
LaunchSite = []
PayloadMass = []
Orbit = []
Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []

# Chiama le funzioni con data_list
getBoosterVersion(data_list)
getLaunchSite(data_list)
getPayloadData(data_list)
getCoreData(data_list)

# Verifica i risultati
print("\n=== RIEPILOGO ===")
print(f"BoosterVersion: {len(BoosterVersion)}")
print(f"LaunchSite: {len(LaunchSite)}")
print(f"PayloadMass: {len(PayloadMass)}")
print(f"Orbit: {len(Orbit)}")
print(f"Block: {len(Block)}")
print(f"Outcome: {len(Outcome)}")

# Mostra i primi 5 risultati
print("\n=== PRIMI 5 RISULTATI ===")
print("BoosterVersion:", BoosterVersion[:5])
print("LaunchSite:", LaunchSite[:5])
print("PayloadMass:", PayloadMass[:5])
print("Orbit:", Orbit[:5])

DataFrame shape: (94, 7)
Tipo di data: <class 'pandas.core.frame.DataFrame'>
Tipo di data_list: <class 'list'>
Numero elementi in data_list: 94
Primo elemento di data_list (esempio):
{'rocket': '5e9d0d95eda69955f709d1eb', 'payloads': '5eb0e4b5b6c3bb0006eeb1e1', 'launchpad': '5e9e4502f5090995de566f86', 'cores': {'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}, 'flight_number': 1, 'date_utc': '2006-03-24T22:30:00.000Z', 'date': datetime.date(2006, 3, 24)}

--- getBoosterVersion ---
Elaborati 94 booster

--- getLaunchSite ---
Elaborati 94 launchpad

--- getPayloadData ---
Elaborati 94 payload

--- getCoreData ---
Elaborati 94 core con dettagli API
Totale core processati: 94

=== RIEPILOGO ===
BoosterVersion: 94
LaunchSite: 94
PayloadMass: 94
Orbit: 94
Block: 94
Outcome: 94

=== PRIMI 5 RISULTATI ===
BoosterVersion: ['Falcon 1', 'Falcon 1', 'Falcon 1

In [52]:
# Dopo aver filtrato il dataframe (data)
print("DataFrame shape:", data.shape)
print("Tipo di data:", type(data))

# Converti il DataFrame in lista di dizionari
data_list = data.to_dict('records')
print("Tipo di data_list:", type(data_list))
print("Numero elementi in data_list:", len(data_list))

# Inizializza le liste
BoosterVersion = []
Longitude = []
Latitude = []
LaunchSite = []
PayloadMass = []
Orbit = []
Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []

# Chiama le funzioni con data_list (NON con data)
print("\n=== CHIAMATA FUNZIONI ===")
getBoosterVersion(data_list)
getLaunchSite(data_list)
getPayloadData(data_list)  # <-- IMPORTANTE: usa data_list
getCoreData(data_list)

# Verifica i risultati
print("\n=== RIEPILOGO ===")
print(f"BoosterVersion: {len(BoosterVersion)}")
print(f"LaunchSite: {len(LaunchSite)}")
print(f"PayloadMass: {len(PayloadMass)}")
print(f"Orbit: {len(Orbit)}")
print(f"Block: {len(Block)}")
print(f"Outcome: {len(Outcome)}")

# Mostra i primi 5 risultati
print("\n=== PRIMI 5 RISULTATI ===")
print("BoosterVersion:", BoosterVersion[:5])
print("LaunchSite:", LaunchSite[:5])
print("PayloadMass:", PayloadMass[:5])
print("Orbit:", Orbit[:5])
print("Block:", Block[:5])
print("Outcome:", Outcome[:5])

DataFrame shape: (94, 7)
Tipo di data: <class 'pandas.core.frame.DataFrame'>
Tipo di data_list: <class 'list'>
Numero elementi in data_list: 94

=== CHIAMATA FUNZIONI ===

--- getBoosterVersion ---
Elaborati 94 booster

--- getLaunchSite ---
Elaborati 94 launchpad

--- getPayloadData ---
Elaborati 94 payload

--- getCoreData ---
Elaborati 94 core con dettagli API
Totale core processati: 94

=== RIEPILOGO ===
BoosterVersion: 94
LaunchSite: 94
PayloadMass: 94
Orbit: 94
Block: 94
Outcome: 94

=== PRIMI 5 RISULTATI ===
BoosterVersion: ['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']
LaunchSite: ['Kwajalein Atoll', 'Kwajalein Atoll', 'Kwajalein Atoll', 'Kwajalein Atoll', 'CCSFS SLC 40']
PayloadMass: [20, None, 165, 200, None]
Orbit: ['LEO', 'LEO', 'LEO', 'LEO', 'LEO']
Block: [None, None, None, None, 1]
Outcome: ['None None', 'None None', 'None None', 'None None', 'None None']


In [53]:
# Chiama tutte le funzioni con data_list
print("=== CHIAMATA FUNZIONI CON DATA_LIST ===")
getBoosterVersion(data_list)
getLaunchSite(data_list)
getPayloadData(data_list)
getCoreData(data_list)  # <-- Usa data_list qui

# Verifica i risultati
print("\n=== RIEPILOGO FINALE ===")
print(f"BoosterVersion: {len(BoosterVersion)}")
print(f"LaunchSite: {len(LaunchSite)}")
print(f"PayloadMass: {len(PayloadMass)}")
print(f"Orbit: {len(Orbit)}")
print(f"Block: {len(Block)}")
print(f"Outcome: {len(Outcome)}")

# Mostra i primi 5 risultati di CoreData
print("\n=== PRIMI 5 COREDATA ===")
for i in range(min(5, len(Block))):
    print(f"Record {i+1}:")
    print(f"  Block: {Block[i]}")
    print(f"  ReusedCount: {ReusedCount[i]}")
    print(f"  Serial: {Serial[i]}")
    print(f"  Outcome: {Outcome[i]}")
    print(f"  Flights: {Flights[i]}")
    print(f"  GridFins: {GridFins[i]}")
    print(f"  Reused: {Reused[i]}")
    print(f"  Legs: {Legs[i]}")
    print(f"  LandingPad: {LandingPad[i]}")
    print()

=== CHIAMATA FUNZIONI CON DATA_LIST ===

--- getBoosterVersion ---
Elaborati 94 booster

--- getLaunchSite ---
Elaborati 94 launchpad

--- getPayloadData ---
Elaborati 94 payload

--- getCoreData ---
Elaborati 94 core con dettagli API
Totale core processati: 188

=== RIEPILOGO FINALE ===
BoosterVersion: 188
LaunchSite: 188
PayloadMass: 188
Orbit: 188
Block: 188
Outcome: 188

=== PRIMI 5 COREDATA ===
Record 1:
  Block: None
  ReusedCount: 0
  Serial: Merlin1A
  Outcome: None None
  Flights: 1
  GridFins: False
  Reused: False
  Legs: False
  LandingPad: None

Record 2:
  Block: None
  ReusedCount: 0
  Serial: Merlin2A
  Outcome: None None
  Flights: 1
  GridFins: False
  Reused: False
  Legs: False
  LandingPad: None

Record 3:
  Block: None
  ReusedCount: 0
  Serial: Merlin2C
  Outcome: None None
  Flights: 1
  GridFins: False
  Reused: False
  Legs: False
  LandingPad: None

Record 4:
  Block: None
  ReusedCount: 0
  Serial: Merlin3C
  Outcome: None None
  Flights: 1
  GridFins: False

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [54]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [56]:
# Create a data from launch_dict
# Opzione 1: Tronca tutte le liste alla lunghezza minima
min_length = min(len(v) for v in launch_dict.values())
print(f"Lunghezza minima: {min_length}")

launch_dict_aligned = {}
for key, value in launch_dict.items():
    launch_dict_aligned[key] = value[:min_length]

df = pd.DataFrame(launch_dict_aligned)
print(f"\nDataframe creato con {min_length} righe")

Lunghezza minima: 94

Dataframe creato con 94 righe


Show the summary of the dataframe


In [57]:
# Show the head of the dataframe
# Mostra le prime 10 righe
print("=== PRIME 10 RIGHE ===")
print(df.head(10))

=== PRIME 10 RIGHE ===
   FlightNumber        Date BoosterVersion  PayloadMass Orbit  \
0             1  2006-03-24       Falcon 1         20.0   LEO   
1             2  2007-03-21       Falcon 1          NaN   LEO   
2             4  2008-09-28       Falcon 1        165.0   LEO   
3             5  2009-07-13       Falcon 1        200.0   LEO   
4             6  2010-06-04       Falcon 9          NaN   LEO   
5             8  2012-05-22       Falcon 9        525.0   LEO   
6            10  2013-03-01       Falcon 9        677.0   ISS   
7            11  2013-09-29       Falcon 9        500.0    PO   
8            12  2013-12-03       Falcon 9       3170.0   GTO   
9            13  2014-01-06       Falcon 9       3325.0   GTO   

        LaunchSite      Outcome  Flights  GridFins  Reused   Legs LandingPad  \
0  Kwajalein Atoll    None None        1     False   False  False       None   
1  Kwajalein Atoll    None None        1     False   False  False       None   
2  Kwajalein Atoll   

### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [58]:
# Hint data['BoosterVersion']!='Falcon 1'
# Filtra il dataframe per mantenere solo i lanci che NON sono Falcon 1
df_filtered = df[df['BoosterVersion'] != 'Falcon 1']

# Verifica il risultato
print(f"Righe originali: {len(df)}")
print(f"Righe dopo filtro (escluso Falcon 1): {len(df_filtered)}")
print(f"Righe rimosse: {len(df) - len(df_filtered)}")

# Mostra le prime righe del dataframe filtrato
print("\n=== PRIME 5 RIGHE (senza Falcon 1) ===")
print(df_filtered.head())

# Controlla i booster rimasti
print("\n=== BOOSTER NEL DATASET FILTRATO ===")
print(df_filtered['BoosterVersion'].value_counts())

Righe originali: 94
Righe dopo filtro (escluso Falcon 1): 90
Righe rimosse: 4

=== PRIME 5 RIGHE (senza Falcon 1) ===
   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite  \
4             6  2010-06-04       Falcon 9          NaN   LEO  CCSFS SLC 40   
5             8  2012-05-22       Falcon 9        525.0   LEO  CCSFS SLC 40   
6            10  2013-03-01       Falcon 9        677.0   ISS  CCSFS SLC 40   
7            11  2013-09-29       Falcon 9        500.0    PO   VAFB SLC 4E   
8            12  2013-12-03       Falcon 9       3170.0   GTO  CCSFS SLC 40   

       Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  \
4    None None        1     False   False  False       None    1.0   
5    None None        1     False   False  False       None    1.0   
6    None None        1     False   False  False       None    1.0   
7  False Ocean        1     False   False  False       None    1.0   
8    None None        1     False   False  False       No

Now that we have removed some values we should reset the FlgihtNumber column


In [60]:
# Prima, crea il dataframe filtrato per Falcon 9 (escludendo Falcon 1)
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()  # .copy() per evitare warning

# Verifica che sia stato creato
print(f"data_falcon9 creato con {len(data_falcon9)} righe")
print("\nPrime 5 righe:")
print(data_falcon9.head())

# Ora puoi resettare il FlightNumber
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))

# Visualizza il risultato
print("\n=== DATAFRAME CON FLIGHTNUMBER RESET ===")
print(data_falcon9[['FlightNumber', 'BoosterVersion', 'Date']].head())

# Verifica i nuovi FlightNumber
print(f"\nNuovi FlightNumber: da {data_falcon9['FlightNumber'].min()} a {data_falcon9['FlightNumber'].max()}")

data_falcon9 creato con 90 righe

Prime 5 righe:
   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite  \
4             6  2010-06-04       Falcon 9          NaN   LEO  CCSFS SLC 40   
5             8  2012-05-22       Falcon 9        525.0   LEO  CCSFS SLC 40   
6            10  2013-03-01       Falcon 9        677.0   ISS  CCSFS SLC 40   
7            11  2013-09-29       Falcon 9        500.0    PO   VAFB SLC 4E   
8            12  2013-12-03       Falcon 9       3170.0   GTO  CCSFS SLC 40   

       Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  \
4    None None        1     False   False  False       None    1.0   
5    None None        1     False   False  False       None    1.0   
6    None None        1     False   False  False       None    1.0   
7  False Ocean        1     False   False  False       None    1.0   
8    None None        1     False   False  False       None    1.0   

   ReusedCount Serial   Longitude   Latitude  
4       

## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [61]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        5
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [62]:
# Calculate the mean value of PayloadMass column

# Replace the np.nan values with its mean value

# Calcola la media della colonna PayloadMass
mean_payload = data_falcon9['PayloadMass'].mean()
print(f"Media del PayloadMass: {mean_payload:.2f} kg")

# Sostituisci i valori NaN con la media
data_falcon9['PayloadMass'].fillna(mean_payload, inplace=True)

# Verifica che non ci siano più NaN
print(f"Valori NaN dopo la sostituzione: {data_falcon9['PayloadMass'].isnull().sum()}")

# Mostra statistiche aggiornate
print("\n=== STATISTICHE PAYLOADMASS DOPO SOSTITUZIONE ===")
print(data_falcon9['PayloadMass'].describe())

Media del PayloadMass: 6123.55 kg
Valori NaN dopo la sostituzione: 0

=== STATISTICHE PAYLOADMASS DOPO SOSTITUZIONE ===
count       90.000000
mean      6123.547647
std       4732.115291
min        350.000000
25%       2510.750000
50%       4701.500000
75%       8912.750000
max      15600.000000
Name: PayloadMass, dtype: float64


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


In [63]:
# Prima di esportare, controlla che il dataframe sia quello giusto
print("=== VERIFICA DATAFRAME ===")
print(f"Nome del dataframe: data_falcon9")
print(f"Dimensioni: {data_falcon9.shape[0]} righe, {data_falcon9.shape[1]} colonne")
print("\nPrime 3 righe:")
print(data_falcon9.head(3))
print("\nColonne:")
print(data_falcon9.columns.tolist())

=== VERIFICA DATAFRAME ===
Nome del dataframe: data_falcon9
Dimensioni: 90 righe, 17 colonne

Prime 3 righe:
   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite  \
4             1  2010-06-04       Falcon 9  6123.547647   LEO  CCSFS SLC 40   
5             2  2012-05-22       Falcon 9   525.000000   LEO  CCSFS SLC 40   
6             3  2013-03-01       Falcon 9   677.000000   ISS  CCSFS SLC 40   

     Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount  \
4  None None        1     False   False  False       None    1.0            0   
5  None None        1     False   False  False       None    1.0            0   
6  None None        1     False   False  False       None    1.0            0   

  Serial  Longitude   Latitude  
4  B0003 -80.577366  28.561857  
5  B0005 -80.577366  28.561857  
6  B0007 -80.577366  28.561857  

Colonne:
['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'Gri

In [2]:
import requests
import pandas as pd

# 1. Effettua la richiesta GET all'API SpaceX per i lanci passati
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)

# 2. Converti la risposta in JSON
launch_data = response.json()

# 3. Crea il dataframe usando json_normalize
df = pd.json_normalize(launch_data)

# 4. Ottieni i dati dei razzi per mappare gli ID ai nomi
rockets_url = "https://api.spacexdata.com/v4/rockets"
rockets_response = requests.get(rockets_url)
rockets_data = rockets_response.json()

# 5. Crea un dizionario per mappare ID razzo -> nome
rocket_names = {rocket['id']: rocket['name'] for rocket in rockets_data}

# 6. Aggiungi una colonna con il nome del razzo
df['rocket_name'] = df['rocket'].map(rocket_names)

# 7. Conta tutti i lanci Falcon 9 (senza filtri aggiuntivi)
total_falcon9 = df[df['rocket_name'] == 'Falcon 9'].shape[0]

print(f"Totale lanci Falcon 9 (tutti): {total_falcon9}")

# 8. Mostra anche i conti per tutti i tipi di razzo
print("\nConta per tipo di razzo:")
print(df['rocket_name'].value_counts())

Totale lanci Falcon 9 (tutti): 179

Conta per tipo di razzo:
rocket_name
Falcon 9        179
Falcon 1          5
Falcon Heavy      3
Name: count, dtype: int64


In [5]:
import requests
import pandas as pd

# Ottieni i dati dei lanci
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# Appiattisci la struttura cores usando json_normalize con record_path
df_cores = pd.json_normalize(
    launch_data,
    record_path=['cores'],
    meta=['flight_number', 'name']
)

# Conta i valori mancanti in 'landpad'
missing_landpad = df_cores['landpad'].isna().sum()
total_cores = len(df_cores)

print(f"Total cores: {total_cores}")
print(f"Missing values in landpad: {missing_landpad}")
print(f"Present values: {total_cores - missing_landpad}")

Total cores: 193
Missing values in landpad: 36
Present values: 157


In [8]:
import requests
import pandas as pd

# === Recupera i dati ===
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# === Inizializza le liste (come nelle tue funzioni) ===
BoosterVersion = []
Longitude = []
Latitude = []
LaunchSite = []
PayloadMass = []
Orbit = []
Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []

# === Funzioni (versione corretta per lavorare con lista di dizionari) ===
def getBoosterVersion(data):
    for launch in data:
        rocket_id = launch.get('rocket')
        if rocket_id:
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/rockets/{rocket_id}").json()
                BoosterVersion.append(response.get('name'))
            except:
                BoosterVersion.append(None)
        else:
            BoosterVersion.append(None)

def getLaunchSite(data):
    for launch in data:
        launchpad_id = launch.get('launchpad')
        if launchpad_id:
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/launchpads/{launchpad_id}").json()
                Longitude.append(response.get('longitude'))
                Latitude.append(response.get('latitude'))
                LaunchSite.append(response.get('name'))
            except:
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append(None)
        else:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)

def getPayloadData(data):
    for launch in data:
        payloads = launch.get('payloads', [])
        if payloads and len(payloads) > 0:
            payload_id = payloads[0]
            try:
                response = requests.get(f"https://api.spacexdata.com/v4/payloads/{payload_id}").json()
                PayloadMass.append(response.get('mass_kg'))
                Orbit.append(response.get('orbit'))
            except:
                PayloadMass.append(None)
                Orbit.append(None)
        else:
            PayloadMass.append(None)
            Orbit.append(None)

def getCoreData(data):
    for launch in data:
        cores = launch.get('cores', [])
        if cores and len(cores) > 0:
            core = cores[0]
            if core.get('core'):
                try:
                    response = requests.get(f"https://api.spacexdata.com/v4/cores/{core['core']}").json()
                    Block.append(response.get('block'))
                    ReusedCount.append(response.get('reuse_count'))
                    Serial.append(response.get('serial'))
                except:
                    Block.append(None)
                    ReusedCount.append(None)
                    Serial.append(None)
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core.get('landing_success')) + ' ' + str(core.get('landing_type')))
            Flights.append(core.get('flight'))
            GridFins.append(core.get('gridfins'))
            Reused.append(core.get('reused'))
            Legs.append(core.get('legs'))
            LandingPad.append(core.get('landpad'))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
            Outcome.append(None)
            Flights.append(None)
            GridFins.append(None)
            Reused.append(None)
            Legs.append(None)
            LandingPad.append(None)

# === Esegui le funzioni ===
print("Esecuzione funzioni di estrazione dati...")
getBoosterVersion(launch_data)
getLaunchSite(launch_data)
getPayloadData(launch_data)
getCoreData(launch_data)

# === Crea il dataframe finale ===
launch_dict = {
    'FlightNumber': list(range(1, len(launch_data) + 1)),
    'Date': [launch.get('date_utc', '')[:10] for launch in launch_data],
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

df_all = pd.DataFrame(launch_dict)

# === FILTRA: escludi Falcon 1 ===
df_falcon9 = df_all[df_all['BoosterVersion'] != 'Falcon 1'].copy()

# === FILTRA: solo lanci con un singolo core e singolo payload ===
df_falcon9 = df_falcon9[df_falcon9['Outcome'].notna()]  # Rimuove righe con Outcome None

# === Calcola i valori mancanti in LandingPad ===
missing_landingpad = df_falcon9['LandingPad'].isna().sum()
total_rows = len(df_falcon9)

print("\n=== RISULTATI ===")
print(f"Total rows in final dataset: {total_rows}")
print(f"Missing values in LandingPad: {missing_landingpad}")
print(f"Present values: {total_rows - missing_landingpad}")

Esecuzione funzioni di estrazione dati...

=== RISULTATI ===
Total rows in final dataset: 182
Missing values in LandingPad: 31
Present values: 151


In [11]:
import requests
import pandas as pd

# Ottieni i dati dall'API
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# Ottieni i nomi dei razzi
rockets_url = "https://api.spacexdata.com/v4/rockets"
rockets_response = requests.get(rockets_url)
rockets_data = rockets_response.json()

rocket_names = {rocket['id']: rocket['name'] for rocket in rockets_data}

# Crea il dataframe
df = pd.json_normalize(launch_data)
df['rocket_name'] = df['rocket'].map(rocket_names)

# CONTO PER TIPO DI RAZZO
print("=== CONTO PER TIPO RAZZO (DATASET API GREZZO) ===")
print(df['rocket_name'].value_counts())
print()

# CONTO TOTALE LANCI
print(f"Totale lanci nel dataset: {len(df)}")
print()

# CONTO FALCON 9 (escludendo Falcon 1)
falcon9_count = df[df['rocket_name'] == 'Falcon 9'].shape[0]
print(f"Falcon 9 launches: {falcon9_count}")

=== CONTO PER TIPO RAZZO (DATASET API GREZZO) ===
rocket_name
Falcon 9        179
Falcon 1          5
Falcon Heavy      3
Name: count, dtype: int64

Totale lanci nel dataset: 187

Falcon 9 launches: 179


In [13]:
import requests
import pandas as pd

# 1. Ottieni i dati dall'API
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# 2. Estrai tutti i core con i rispettivi landing pad
# Questo è ciò che viene raccolto durante il "API data collection process"
cores_list = []

for launch in launch_data:
    for core in launch.get('cores', []):
        cores_list.append({
            'flight_number': launch.get('flight_number'),
            'core_serial': core.get('core'),
            'landpad': core.get('landpad'),
            'landing_success': core.get('landing_success'),
            'landing_type': core.get('landing_type')
        })

df_cores = pd.DataFrame(cores_list)

# 3. Conta i valori mancanti in landpad
missing_landpad = df_cores['landpad'].isna().sum()
total_cores = len(df_cores)

print(f"=== DATASET CORES (dopo API data collection) ===")
print(f"Total cores: {total_cores}")
print(f"Missing values in landpad: {missing_landpad}")
print(f"Present values: {total_cores - missing_landpad}")

=== DATASET CORES (dopo API data collection) ===
Total cores: 193
Missing values in landpad: 36
Present values: 157


In [14]:
import requests
import pandas as pd

# Ottieni i dati
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
launch_data = response.json()

# Ottieni i nomi dei razzi
rockets_url = "https://api.spacexdata.com/v4/rockets"
rockets_response = requests.get(rockets_url)
rockets_data = rockets_response.json()

rocket_names = {rocket['id']: rocket['name'] for rocket in rockets_data}

# Crea dataframe
df = pd.json_normalize(launch_data)
df['rocket_name'] = df['rocket'].map(rocket_names)

# Risposta 1: solo Falcon 9
falcon9_only = df[df['rocket_name'] == 'Falcon 9'].shape[0]
print(f"Solo Falcon 9: {falcon9_only}")

# Risposta 2: totale dopo rimozione Falcon 1
total_no_falcon1 = df[df['rocket_name'] != 'Falcon 1'].shape[0]
print(f"Totale lanci (escluso Falcon 1): {total_no_falcon1}")

# Risposta 3: totale lanci (tutti)
print(f"Totale lanci: {len(df)}")

Solo Falcon 9: 179
Totale lanci (escluso Falcon 1): 182
Totale lanci: 187


In [16]:
# Verifica se nel dataset finale ci sono solo Falcon 9 o anche Falcon Heavy
if 'df_falcon9' in dir():
    print("Valori unici in BoosterVersion:")
    print(df_falcon9['BoosterVersion'].unique())
    print(f"\nConteggi:")
    print(df_falcon9['BoosterVersion'].value_counts())
else:
    print("df_falcon9 non esiste")

Valori unici in BoosterVersion:
['Falcon 9' 'Falcon Heavy']

Conteggi:
BoosterVersion
Falcon 9        179
Falcon Heavy      3
Name: count, dtype: int64


In [17]:
# Se data_falcon9 esiste dal tuo codice precedente
try:
    print("=== DATASET data_falcon9 (con tutti i filtri) ===")
    print(f"Shape: {data_falcon9.shape}")
    print(f"BoosterVersion unici: {data_falcon9['BoosterVersion'].unique()}")
    print(f"\nConteggi per BoosterVersion:")
    print(data_falcon9['BoosterVersion'].value_counts())
    
except NameError:
    print("data_falcon9 non esiste")
    
    # Prova a ricaricare dal CSV se esiste
    try:
        data_falcon9 = pd.read_csv('dataset_part_1.csv')
        print("\n=== DATASET CARICATO DA CSV ===")
        print(f"Shape: {data_falcon9.shape}")
        print(f"BoosterVersion unici: {data_falcon9['BoosterVersion'].unique()}")
        print(f"\nConteggi per BoosterVersion:")
        print(data_falcon9['BoosterVersion'].value_counts())
    except:
        print("File dataset_part_1.csv non trovato")

=== DATASET data_falcon9 (con tutti i filtri) ===
data_falcon9 non esiste

=== DATASET CARICATO DA CSV ===
Shape: (90, 17)
BoosterVersion unici: ['Falcon 9']

Conteggi per BoosterVersion:
BoosterVersion
Falcon 9    90
Name: count, dtype: int64


In [18]:
import pandas as pd

# Carica il dataset finale
data_falcon9 = pd.read_csv('dataset_part_1.csv')

# Calcola i valori mancanti nella colonna LandingPad
missing_landingpad = data_falcon9['LandingPad'].isna().sum()
total_rows = len(data_falcon9)

print(f"=== DATASET FINALE (dataset_part_1.csv) ===")
print(f"Total rows: {total_rows}")
print(f"Missing values in LandingPad: {missing_landingpad}")
print(f"Present values: {total_rows - missing_landingpad}")

# Mostra anche i primi valori per verifica
print(f"\nPrime 10 righe della colonna LandingPad:")
print(data_falcon9['LandingPad'].head(10))

=== DATASET FINALE (dataset_part_1.csv) ===
Total rows: 90
Missing values in LandingPad: 26
Present values: 64

Prime 10 righe della colonna LandingPad:
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
Name: LandingPad, dtype: object


In [4]:
from bs4 import BeautifulSoup
import requests

# URL della pagina
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Ottieni la risposta
response = requests.get(static_url, headers=headers)

# Crea l'oggetto BeautifulSoup
soup = BeautifulSoup(response.content, 'html.parser')

# Stampa soup.title
print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
